# Ensemble: V5.0 + V6.0 Softmax Averaging

In [ ]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy numpy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'Data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')
print('Setup complete')

## Ensemble: V5.0 + V6.0

V5.0 (Multi-scale SeqCA): ET=0.7910 强, TC/WT 较弱
V6.0 (Bottleneck SeqCA): TC=0.8646, WT=0.8991 强, ET 较弱

策略: 两个模型分别推理, softmax 概率平均后 argmax。零训练成本。

In [6]:
# Ensemble V5.0 + V6.0: softmax averaging
import os, sys, re, yaml, torch, numpy as np
from pathlib import Path
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Fix fusion.py for bottleneck-only mode (V6.0 needs this)
with open('models/fusion.py', 'r') as f:
    code = f.read()
if 'bottleneck_only' not in code:
    old = '        return [\n'
    old += '            attn(feat, text_feat, text_mask)\n'
    old += '            for attn, feat in zip(self.attn_layers, features)\n'
    old += '        ]'
    new = '        if len(features) < len(self.attn_layers):\n'
    new += '            offset = len(self.attn_layers) - len(features)\n'
    new += '            return [\n'
    new += '                attn(feat, text_feat, text_mask)\n'
    new += '                for attn, feat in zip(self.attn_layers[offset:], features)\n'
    new += '            ]\n'
    new += '        return [\n'
    new += '            attn(feat, text_feat, text_mask)\n'
    new += '            for attn, feat in zip(self.attn_layers, features)\n'
    new += '        ]'
    code = code.replace(old, new)
    with open('models/fusion.py', 'w') as f:
        f.write(code)
    print('Patched fusion.py for bottleneck-only')

# Patch textmamba3d.py if needed
with open('models/textmamba3d.py', 'r') as f:
    code = f.read()
if 'fusion_mode' not in code:
    code = code.replace(
        "fusion_type: str = \"seqca\",",
        "fusion_type: str = \"seqca\",\n"
        "        fusion_mode: str = \"multi_scale\","
    )
    code = code.replace(
        "self.multi_scale_attn = fusion_cls(",
        "self.fusion_mode = fusion_mode\n"
        "        self.multi_scale_attn = fusion_cls("
    )
    old_fuse = '''            fused = self.multi_scale_attn(
                img_features[1:], text_features, attention_mask
            )'''
    new_fuse = '''            if self.fusion_mode == "bottleneck_only":
                shallow = img_features[1:-1]
                deep = [img_features[-1]]
                fused_deep = self.multi_scale_attn(deep, text_features, attention_mask)
                if self.text_gate is not None:
                    fused_deep = self.text_gate(deep, fused_deep)
                fused = list(shallow) + list(fused_deep)
            else:
                fused = self.multi_scale_attn(
                    img_features[1:], text_features, attention_mask
                )'''
    code = code.replace(old_fuse, new_fuse)
    with open('models/textmamba3d.py', 'w') as f:
        f.write(code)
    print('Patched textmamba3d.py for fusion_mode')

# Patch evaluate_full.py if needed
with open('evaluate_full.py', 'r') as f:
    code = f.read()
if 'fusion_mode' not in code:
    code = code.replace(
        "fusion_type=model_cfg.get('fusion_type', 'seqca'),",
        "fusion_type=model_cfg.get('fusion_type', 'seqca'),\n"
        "        fusion_mode=model_cfg.get('fusion_mode', 'multi_scale'),"
    )
    with open('evaluate_full.py', 'w') as f:
        f.write(code)
    print('Patched evaluate_full.py')

print('All patches applied')

In [7]:
# Step 1: Save softmax predictions from V5.0
import os
os.chdir(REPO_DIR)

V50_CKPT = os.path.join(DRIVE_CKPT, 'best_v5.0.pth')
V60_CKPT = os.path.join(DRIVE_CKPT, 'best_V6.0.pth')
assert os.path.exists(V50_CKPT), f'V5.0 checkpoint not found: {V50_CKPT}'
assert os.path.exists(V60_CKPT), f'V6.0 checkpoint not found: {V60_CKPT}'

PRED_DIR = os.path.join(DRIVE_BASE, 'ensemble_preds')
os.makedirs(f'{PRED_DIR}/v50', exist_ok=True)
os.makedirs(f'{PRED_DIR}/v60', exist_ok=True)

print('Saving V5.0 predictions (text+TTA)...')
!python -u evaluate_full.py \
    --config configs/archive/textbrats_a100_v5.yaml \
    --checkpoint "{V50_CKPT}" \
    --split test --overlap 0.5 \
    --use-text --tta \
    --save-preds "{PRED_DIR}/v50"

print('\nV5.0 predictions saved')

In [8]:
# Step 2: Save softmax predictions from V6.0
import os
os.chdir(REPO_DIR)

# Generate V6.0 config if not exists
import yaml
if not os.path.exists('configs/autoresearch/V6.0_bottleneck_seqca.yaml'):
    os.makedirs('configs/autoresearch', exist_ok=True)
    with open('configs/archive/textbrats_a100_v5.yaml') as f:
        base = yaml.safe_load(f)
    base['model']['fusion_mode'] = 'bottleneck_only'
    base['training']['gradient_checkpointing'] = False
    with open('configs/autoresearch/V6.0_bottleneck_seqca.yaml', 'w') as f:
        yaml.dump(base, f, default_flow_style=False, sort_keys=False)

print('Saving V6.0 predictions (text+TTA)...')
!python -u evaluate_full.py \
    --config configs/autoresearch/V6.0_bottleneck_seqca.yaml \
    --checkpoint "{V60_CKPT}" \
    --split test --overlap 0.5 \
    --use-text --tta \
    --save-preds "{PRED_DIR}/v60"

print('\nV6.0 predictions saved')

In [9]:
# Step 3: Ensemble — average softmax probabilities
import os, numpy as np, re
os.chdir(REPO_DIR)

PRED_DIR = os.path.join(DRIVE_BASE, 'ensemble_preds')
v50_dir = f'{PRED_DIR}/v50'
v60_dir = f'{PRED_DIR}/v60'

v50_files = sorted([f for f in os.listdir(v50_dir) if f.endswith('_probs.npy')])
v60_files = sorted([f for f in os.listdir(v60_dir) if f.endswith('_probs.npy')])

print(f'V5.0 predictions: {len(v50_files)}')
print(f'V6.0 predictions: {len(v60_files)}')

# Match by case name
common = set(v50_files) & set(v60_files)
print(f'Common cases: {len(common)}')

from utils.metrics import dice_score_brats_regions
import nibabel as nib

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

# Try different ensemble weights
weights_to_try = [
    ('equal', 0.5, 0.5),
    ('v50_heavy', 0.6, 0.4),
    ('v60_heavy', 0.4, 0.6),
    ('v50_et_focus', 0.7, 0.3),
]

for weight_name, w50, w60 in weights_to_try:
    all_dice_et, all_dice_tc, all_dice_wt = [], [], []

    for fname in sorted(common):
        case_name = fname.replace('_probs.npy', '')
        probs_v50 = np.load(os.path.join(v50_dir, fname))
        probs_v60 = np.load(os.path.join(v60_dir, fname))

        # Weighted average of softmax probabilities
        ensemble_probs = w50 * probs_v50 + w60 * probs_v60
        pred = np.argmax(ensemble_probs, axis=0)

        # Load ground truth
        seg_path = os.path.join(DATA_DIR, case_name, f'{case_name}_seg.nii')
        if not os.path.exists(seg_path):
            seg_path = seg_path + '.gz'
        gt = nib.load(seg_path).get_fdata().astype(np.int64)
        # BraTS label mapping: 0=bg, 1=NCR, 2=ED, 4=ET -> remap 4->3
        gt[gt == 4] = 3

        pred_t = torch.from_numpy(ensemble_probs).unsqueeze(0).float()
        gt_t = torch.from_numpy(gt).unsqueeze(0).long()
        dice = dice_score_brats_regions(pred_t, gt_t)

        all_dice_et.append(dice['dice_ET'])
        all_dice_tc.append(dice['dice_TC'])
        all_dice_wt.append(dice['dice_WT'])

    et = np.mean(all_dice_et)
    tc = np.mean(all_dice_tc)
    wt = np.mean(all_dice_wt)
    mean = (et + tc + wt) / 3

    print(f'\n{weight_name} (w50={w50}, w60={w60}):')
    print(f'  ET={et:.4f}  TC={tc:.4f}  WT={wt:.4f}  Mean={mean:.4f}')

In [11]:
# Fine-grained ensemble weight search
import os, numpy as np, torch
os.chdir(REPO_DIR)
from utils.metrics import dice_score_brats_regions
import nibabel as nib

PRED_DIR = os.path.join(DRIVE_BASE, 'ensemble_preds')
v50_dir = f'{PRED_DIR}/v50'
v60_dir = f'{PRED_DIR}/v60'
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

common = sorted(set(os.listdir(v50_dir)) & set(os.listdir(v60_dir)))
common = [f for f in common if f.endswith('_probs.npy')]
print(f'Cases: {len(common)}')

# Search weights from 0.5 to 0.9 in 0.05 steps
best_mean = 0
best_w = 0
results = []

for w50 in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    w60 = 1.0 - w50
    all_et, all_tc, all_wt = [], [], []
    for fname in common:
        case = fname.replace('_probs.npy', '')
        p50 = np.load(os.path.join(v50_dir, fname))
        p60 = np.load(os.path.join(v60_dir, fname))
        ens = w50 * p50 + w60 * p60
        seg_path = os.path.join(DATA_DIR, case, f'{case}_seg.nii')
        if not os.path.exists(seg_path):
            seg_path += '.gz'
        gt = nib.load(seg_path).get_fdata().astype(np.int64)
        gt[gt == 4] = 3
        pred_t = torch.from_numpy(ens).unsqueeze(0).float()
        gt_t = torch.from_numpy(gt).unsqueeze(0).long()
        d = dice_score_brats_regions(pred_t, gt_t)
        all_et.append(d['dice_ET'])
        all_tc.append(d['dice_TC'])
        all_wt.append(d['dice_WT'])
    et = np.mean(all_et)
    tc = np.mean(all_tc)
    wt = np.mean(all_wt)
    mean = (et + tc + wt) / 3
    results.append((w50, et, tc, wt, mean))
    if mean > best_mean:
        best_mean = mean
        best_w = w50
    print(f'w50={w50:.2f}: ET={et:.4f} TC={tc:.4f} WT={wt:.4f} Mean={mean:.4f}')

print(f'\nBest: w50={best_w:.2f}, Mean={best_mean:.4f}')
print(f'Baseline V5.0: Mean=0.8479')
print(f'Delta: {best_mean - 0.8479:+.4f}')

## Results

In [10]:
# Summary comparison
print('=' * 70)
print('ENSEMBLE RESULTS vs INDIVIDUAL MODELS')
print('=' * 70)
print(f'{"Model":<25} {"ET":>8} {"TC":>8} {"WT":>8} {"Mean":>8}')
print('-' * 58)
print(f'{"V5.0 (multi-scale)":<25} {"0.7910":>8} {"0.8560":>8} {"0.8967":>8} {"0.8479":>8}')
print(f'{"V6.0 (bottleneck)":<25} {"0.7770":>8} {"0.8646":>8} {"0.8991":>8} {"0.8469":>8}')
print(f'{"TextBraTS SOTA":<25} {"0.8330":>8} {"0.8280":>8} {"0.8990":>8} {"0.8530":>8}')
print('-' * 58)
print('Ensemble results shown above ^^')
print()
print('Target: Mean > 0.8479 (V5.0 baseline)')